# SeraTune TRIBE v2 Worker

**3 cells, no restarts needed.** Run them top to bottom.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** (or A100 with Colab Pro)
- Have your [HuggingFace token](https://huggingface.co/settings/tokens) ready (accept [LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B) first)
- Have your [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken) ready

In [ ]:
# ============================================================
# CELL 1: Install everything
# ============================================================
!pip uninstall -y torch torchaudio torchvision numpy neuralset neuraltrain tribev2 click gtts 2>/dev/null

!pip install -q \
  "numpy==2.2.6" \
  "torch==2.6.0" \
  "torchaudio==2.6.0" \
  "torchvision==0.21.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  "neuralset==0.0.2" \
  "neuraltrain==0.0.2" \
  "x_transformers==1.27.20" \
  "moviepy>=2.2.1" \
  gtts julius Levenshtein transformers huggingface_hub \
  soundfile librosa langdetect spacy

!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q fastapi uvicorn pyngrok python-multipart httpx

print('\n✅ Packages installed. Run the next cell.')

In [ ]:
# ============================================================
# CELL 2: Start the worker (runs in a fresh subprocess
#          so it gets the new numpy/torch — no restart needed)
# ============================================================

HF_TOKEN = ""       # paste your HuggingFace token here
NGROK_TOKEN = ""    # paste your ngrok auth token here

if not HF_TOKEN:
    HF_TOKEN = input('Enter your HuggingFace token: ')
if not NGROK_TOKEN:
    NGROK_TOKEN = input('Enter your ngrok auth token: ')

worker_script = '''
import sys, os, logging, tempfile, time
import numpy as np
import torch
from huggingface_hub import login
from tribev2 import TribeModel
from fastapi import FastAPI, File, HTTPException, UploadFile
import uvicorn
from pyngrok import ngrok

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Auth
hf_token = os.environ.get("HF_TOKEN", "")
ngrok_token = os.environ.get("NGROK_TOKEN", "")
if hf_token:
    login(token=hf_token)
else:
    print("ERROR: HF_TOKEN not set"); sys.exit(1)
if not ngrok_token:
    print("ERROR: NGROK_TOKEN not set"); sys.exit(1)

# GPU info
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A"
print(f"GPU: {gpu} ({vram})")

# Load model
print("Loading TRIBE v2 model...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded!")

# FastAPI app
app = FastAPI(title="TRIBE v2 Worker")

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": True}

@app.post("/analyze")
async def analyze(audio: UploadFile = File(...)):
    tmp_path = None
    try:
        suffix = os.path.splitext(audio.filename or "audio.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp_path = tmp.name
            content = await audio.read()
            tmp.write(content)
        logger.info("Running inference on %s (%d bytes)...", audio.filename, len(content))
        start = time.time()
        df = model.get_events_dataframe(audio_path=tmp_path)
        preds, segments = model.predict(events=df)
        elapsed = time.time() - start
        logger.info("Inference complete: %s -> %s in %.1fs", audio.filename, preds.shape, elapsed)
        return {
            "preds": preds.tolist(),
            "shape": list(preds.shape),
            "n_segments": preds.shape[0],
            "n_vertices": preds.shape[1],
            "inference_time_s": round(elapsed, 2),
        }
    except Exception as e:
        logger.exception("Inference failed")
        raise HTTPException(500, f"Inference failed: {e}")
    finally:
        if tmp_path:
            os.unlink(tmp_path)

# Start ngrok + uvicorn
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8001).public_url

print()
print("=" * 60)
print(f"TRIBE v2 Worker is running!")
print(f"")
print(f"  Public URL: {public_url}")
print(f"")
print(f"  Add to your backend .env:")
print(f"    TRIBE_WORKER_URL={public_url}")
print(f"    USE_MOCK_TRIBE=false")
print("=" * 60)
sys.stdout.flush()

uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")
'''

with open('/tmp/tribe_worker_full.py', 'w') as f:
    f.write(worker_script)

import subprocess, os
env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN
env['NGROK_TOKEN'] = NGROK_TOKEN

print('Starting TRIBE worker in a fresh process...')
print('(Model loads + server starts — takes a few minutes)\n')

# Run as subprocess — fresh Python process, no stale numpy
proc = subprocess.Popen(
    ['python', '/tmp/tribe_worker_full.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream output to notebook
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
    print('\nWorker stopped.')

In [ ]:
# ============================================================
# CELL 3 (optional): If the worker dies, re-run Cell 2.
# This cell is just for notes.
# ============================================================
print('If the worker stopped, just re-run Cell 2.')
print('The model weights are cached in ./cache so reloading is fast.')